# 1. HOG (Histogram of Oriented Gradients) para Clasificación de Imágenes
### Clasificación de Categorías de Imágenes: Manzanas, Perros y Personas

Este cuaderno implementa la solución completa de **Image Detection / Classification** utilizando el descriptor **HOG** y un clasificador **SVM** con OpenCV:
- **Entrada**: Una imagen cualquiera.
- **Salida**: La clase predicha a la que pertenece la imagen (`apples`, `dogs` o `people`).

#### Correcciones clave respecto al trabajo previo en `mine/hog.ipynb`:
1. **Soporte de extensiones `.jpeg`**: El código original ignoraba las imágenes `.jpeg`, omitiendo el 100% de las manzanas (164 de train y 54 de test). Aquí se cargan todas las imágenes dinámicamente.
2. **Dimensionamiento óptimo de ventana**: Se sustituye la ventana de $512 \times 512$ (que producía 142,884 dimensiones y un modelo de 628 MB) por una ventana de $128 \times 128$ (8,100 dimensiones, modelo de 39 MB, entrenamiento instantáneo).
3. **Evaluación completa en el conjunto de prueba**: Cálculo de exactitud, reporte de clasificación por clase y matriz de confusión.



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from dataset import load_split
from hog_pipeline import HOGPipeline

def show_image(img, title=None):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if len(img.shape) == 3 else img
    plt.figure(figsize=(5, 5))
    plt.imshow(img_rgb)
    plt.axis('off')
    if title:
        plt.title(title, fontsize=12, fontweight='bold')
    plt.show()



## Paso 1: Carga de Datos de Entrenamiento y Prueba


In [ ]:
train_imgs, train_lbls, classes, _ = load_split("train")
test_imgs, test_lbls, _, _ = load_split("test")

print(f"Clases detectadas: {classes}")
print(f"Muestras de entrenamiento: {len(train_imgs)}")
print(f"Muestras de prueba:        {len(test_imgs)}")

for c in classes:
    print(f"  - {c}: {train_lbls.count(c)} train, {test_lbls.count(c)} test")



## Paso 2: Configuración del Descriptor HOG
Ventana: `(128, 128)`, Bloque: `(16, 16)`, Paso de bloque: `(8, 8)`, Celda: `(8, 8)`, Contenedores: `9`.
Dimensión resultante: $15 \times 15 \times 36 = 8,100$ características.



In [ ]:
pipeline = HOGPipeline(win_size=(128, 128))
sample_des = pipeline.compute_descriptor(train_imgs[0])
print(f"Dimensión del descriptor HOG por imagen: {sample_des.shape[0]} características flotantes")



## Paso 3: Entrenamiento del Clasificador SVM
Entrenamos la SVM de OpenCV (`cv2.ml.SVM`) con kernel lineal para clasificar las imágenes.



In [ ]:
print("Entrenando clasificador HOG + SVM...")
train_info = pipeline.train(train_imgs, train_lbls, kernel_type="linear", C=1.0)
print(f"Entrenamiento completado en {train_info['train_time_sec']:.2f} segundos!")



## Paso 4: Evaluación en el Conjunto de Prueba
Evaluamos la exactitud, precisión, recall y matriz de confusión en las 140 imágenes de prueba.



In [ ]:
eval_results = pipeline.evaluate(test_imgs, test_lbls)
print(f"Exactitud en Prueba (Accuracy): {eval_results['accuracy'] * 100:.2f}%
")
print("Reporte de Clasificación por Clase:")
print(classification_report(test_lbls, eval_results['predictions'], target_names=pipeline.classes_))

# Matriz de confusión
cm = np.array(eval_results["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes, fontsize=11)
ax.set_yticklabels(classes, fontsize=11)
plt.xlabel("Clase Predicha", fontsize=11)
plt.ylabel("Clase Real", fontsize=11)
plt.title("Matriz de Confusión HOG + SVM", fontsize=12, fontweight='bold')
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="red" if i != j else "black", fontweight='bold')
plt.colorbar(im)
plt.tight_layout()
plt.show()



## Paso 5: Demostración de Clasificación en Imágenes Individuales
Dada una imagen de entrada, el modelo predice directamente su clase semántica.



In [ ]:
import random

# Seleccionamos muestras aleatorias de prueba
for _ in range(3):
    idx = random.randrange(len(test_imgs))
    img = test_imgs[idx]
    true_label = test_lbls[idx]
    
    pred_label, _ = pipeline.predict([img])
    show_image(img, title=f"Real: {true_label.upper()} | Predicción HOG: {pred_label[0].upper()}")



## Paso 6: Guardado del Modelo


In [ ]:
pipeline.save_model("hog_svm_model.yml")
model_size_mb = Path("hog_svm_model.yml").stat().st_size / (1024 * 1024)
print(f"Modelo guardado: hog_svm_model.yml ({model_size_mb:.2f} MB)")

